# 04 Experiment Sandbox: Hyperparameter Workbench

Use this notebook to test variants before committing settings to YAML. The intended workflow is:

1. Run the official pipeline only up to the upstream artifact needed for an experiment.
2. Run the relevant experiment section here.
3. Inspect the compact `*_summary.csv` or `*_summary.md` in `outputs/<mode>/experiments/<experiment_name>/`.
4. Promote only the winning hyperparameters into `configs/dev.yaml` or `configs/base.yaml`.
5. Rerun `03_ml_pipeline.ipynb` as the clean official pipeline.

Sandbox outputs are evidence, not final deliverables.


## Setup

Set `RUN_MODE`, load the prepared transaction scan, and point the notebook at existing official pipeline artifacts. Missing artifacts are expected if you have not run the required upstream stage yet.


In [1]:
import os
from pathlib import Path
import sys

from IPython.display import Markdown, display

# Mode toggle: experiments should usually run in dev first.
RUN_MODE = "dev"  # change to "prod" only when intentionally experimenting on full data
os.environ["CARREFOUR_MODE"] = RUN_MODE.strip().lower()

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break
else:
    project_root = Path.cwd().resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import configure_mode
from src.data_loader import load_prepared_transactions
from src.utils import set_global_seed

CONFIG = configure_mode(RUN_MODE)
MODE = CONFIG.mode
DATA_PROCESSED = CONFIG.data_processed
OUTPUTS = CONFIG.outputs

set_global_seed(CONFIG.random_seed)
CONFIG.ensure_directories()
transactions = load_prepared_transactions(cfg=CONFIG)

basket_path = CONFIG.artifact_path("baskets", "output", directory=CONFIG.outputs / "embeddings")
product_embeddings_path = CONFIG.artifact_path("word2vec", "embeddings_output", directory=CONFIG.outputs / "embeddings")
customer_embeddings_path = CONFIG.artifact_path("customer_embeddings", "output", directory=CONFIG.outputs / "features")
behavior_path = CONFIG.artifact_path("behavioral_features", "output", directory=CONFIG.outputs / "features")
feature_sets = {
    name: CONFIG.outputs / "features" / filename
    for name, filename in CONFIG.get("feature_sets.outputs", {}).items()
}

artifact_status = {
    "basket_path": basket_path,
    "product_embeddings_path": product_embeddings_path,
    "customer_embeddings_path": customer_embeddings_path,
    "behavior_path": behavior_path,
    **{f"feature_set:{name}": path for name, path in feature_sets.items()},
}

print(f"Run mode: {MODE}")
print(f"Data path: {DATA_PROCESSED}")
print(f"Experiment output path: {CONFIG.experiments}")
print(f"Experiments enabled: {CONFIG.experiments_enabled}")
if not CONFIG.experiments_enabled:
    display(Markdown("**Experiments are disabled for this mode.** Switch `RUN_MODE` to `dev` for sandbox runs."))
for label, path in artifact_status.items():
    status = "exists" if path.exists() else "missing"
    print(f"{label}: {status} | {path}")


Run mode: dev
Data path: C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\data\dev
Experiment output path: C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments
Experiments enabled: True
basket_path: exists | C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\embeddings\basket_sentences.parquet
product_embeddings_path: exists | C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\embeddings\product_embeddings.parquet
customer_embeddings_path: exists | C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\features\customer_embeddings.parquet
behavior_path: exists | C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\features\customer_behavior_features.parquet
feature_set:embeddings_only: exists | C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\features\feature_set_embeddings_only.parquet
feature_set:embeddings_behavior: exist

## Experiment 2: Item2Vec Training

Optional sandbox for training Item2Vec variants without changing YAML. Edit the trial list in the cell below. This stage only trains models and writes product embedding tables; Experiment 3 evaluates those tables.

In [2]:
# Evaluate window size
import polars as pl
from IPython.display import Image, Markdown, display

from src.experiment_sandbox import run_item2vec_training_experiments

RUN_ITEM2VEC_TRAINING_SANDBOX = True
item2vec_sandbox_experiment_name = "item2vec_window_sandbox"
item2vec_sandbox_trials = [
    {
        "name": "window5",
        "word2vec": {"window": 5},
    },
    {
        "name": "window10",
        "word2vec": {"window": 10},
    },
    {
        "name": "window15",
        "word2vec": {"window": 15},
    },
    {
        "name": "window20",
        "word2vec": {"window": 20},
    },
]

if RUN_ITEM2VEC_TRAINING_SANDBOX:
    item2vec_training_sandbox = run_item2vec_training_experiments(
        basket_path,
        trials=item2vec_sandbox_trials,
        experiment_name=item2vec_sandbox_experiment_name,
        force=False,
        cfg=CONFIG,
    )
    display(Markdown(f"""
### Item2Vec Training Experiment

- Manifest parquet: `{item2vec_training_sandbox['manifest']['parquet']}`
- Manifest summary CSV: `{item2vec_training_sandbox['manifest']['summary_csv']}`
- Manifest summary Markdown: `{item2vec_training_sandbox['manifest']['summary_md']}`
"""))
    display(pl.read_parquet(item2vec_training_sandbox["manifest"]["parquet"]))
else:
    display(Markdown(
        "Item2Vec training sandbox is self-contained and currently disabled. "
        "Set `RUN_ITEM2VEC_TRAINING_SANDBOX = True` in this cell to train variants."
    ))

[01:21:32] Item2Vec training sandbox: training Item2Vec trials | experiment=item2vec_window_sandbox, trials=4
[01:21:32] Item2Vec training sandbox: starting trial | experiment=item2vec_window_sandbox, trial=window5, overrides={"window": 5}
Item2Vec: training model
  baskets: 788,724
  product tokens: 7,511,295; avg/basket: 9.52; median/basket: 6.00
  model path: /Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/window5_word2vec_product.model
  config: vector_size=64, window=190 (configured=5, full_basket_context=True), min_count=10, negative=10, sample=0.001, epochs=3, sg=1, workers=1, seed=42
Item2Vec epoch 1/3 started


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Item2Vec epoch 1/3 finished in 205.3s; elapsed=208.1s
Item2Vec epoch 2/3 started


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Item2Vec epoch 2/3 finished in 204.1s; elapsed=412.2s
Item2Vec epoch 3/3 started


...

Item2Vec epoch 2/3 finished in 202.4s; elapsed=407.5s
Item2Vec epoch 3/3 started


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Item2Vec epoch 3/3 finished in 203.3s; elapsed=610.8s
Item2Vec model saved: vocab=37,299, vector_size=64, path=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/window20_word2vec_product.model
[02:02:35] Stage 2 product embeddings: wrote artifact | products=37299, path=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/window20_product_embeddings.parquet
[02:02:35] Item2Vec training sandbox: finished | elapsed_s=2464.0000
[02:02:36] Experiment summary: wrote human-readable summaries | rows=4, csv=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_item2vec_training_summary.csv, markdown=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_item2vec_training_summary.md
[02:02:36] Experiment sandbox: wrote manifest 


### Item2Vec Training Experiment

- Manifest parquet: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_item2vec_training_manifest.parquet`
- Manifest summary CSV: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_item2vec_training_summary.csv`
- Manifest summary Markdown: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_item2vec_training_summary.md`


experiment_name,trial_name,word2vec_overrides,model_path,product_embeddings_path
str,str,str,str,str
"""item2vec_window_sandbox""","""window5""","""{""window"": 5}""","""/Users/ljyr/Documents/IE/Capst…","""/Users/ljyr/Documents/IE/Capst…"
"""item2vec_window_sandbox""","""window10""","""{""window"": 10}""","""/Users/ljyr/Documents/IE/Capst…","""/Users/ljyr/Documents/IE/Capst…"
"""item2vec_window_sandbox""","""window15""","""{""window"": 15}""","""/Users/ljyr/Documents/IE/Capst…","""/Users/ljyr/Documents/IE/Capst…"
"""item2vec_window_sandbox""","""window20""","""{""window"": 20}""","""/Users/ljyr/Documents/IE/Capst…","""/Users/ljyr/Documents/IE/Capst…"


In [4]:
# Tests different Item2Vec training parameters
import polars as pl
from IPython.display import Image, Markdown, display

from src.experiment_sandbox import run_item2vec_training_experiments

RUN_ITEM2VEC_TRAINING_SANDBOX = True
item2vec_sandbox_experiment_name = "item2vec_params_sandbox"
item2vec_sandbox_trials = [
    {
        "name": "current_config",
        "word2vec": {},
    },
    {
        "name": "dims100_epochs5",
        "word2vec": {
            "vector_size": 100,
            "epochs": 5,
        },
    },
    {
        "name": "dims128_epochs5",
        "word2vec": {
            "vector_size": 128,
            "epochs": 5,
        },
    },
    {
        "name": "dims100_min_count5",
        "word2vec": {
            "vector_size": 100,
            "epochs": 5,
            "min_count": 5,
        },
    },
    {
        "name": "dims100_subsample0005",
        "word2vec": {
            "vector_size": 100,
            "epochs": 5,
            "sample": 0.0005,
        },
    },
    {
        "name": "dims100_min_count5_subsample0005",
        "word2vec": {
            "vector_size": 100,
            "epochs": 5,
            "min_count": 5,
            "sample": 0.0005,
        },
    },
    {
        "name": "dims100_negative15",
        "word2vec": {
            "vector_size": 100,
            "epochs": 5,
            "negative": 15,
        },
    },
]

if RUN_ITEM2VEC_TRAINING_SANDBOX:
    item2vec_training_sandbox = run_item2vec_training_experiments(
        basket_path,
        trials=item2vec_sandbox_trials,
        experiment_name=item2vec_sandbox_experiment_name,
        force=False,
        cfg=CONFIG,
    )
    display(Markdown(f"""
### Item2Vec Training Experiment

- Manifest parquet: `{item2vec_training_sandbox['manifest']['parquet']}`
- Manifest summary CSV: `{item2vec_training_sandbox['manifest']['summary_csv']}`
- Manifest summary Markdown: `{item2vec_training_sandbox['manifest']['summary_md']}`
"""))
    display(pl.read_parquet(item2vec_training_sandbox["manifest"]["parquet"]))
else:
    display(Markdown(
        "Item2Vec training sandbox is self-contained and currently disabled. "
        "Set `RUN_ITEM2VEC_TRAINING_SANDBOX = True` in this cell to train variants."
    ))

[02:33:09] Item2Vec training sandbox: training Item2Vec trials | experiment=item2vec_params_sandbox, trials=7
[02:33:09] Item2Vec training sandbox: starting trial | experiment=item2vec_params_sandbox, trial=current_config, overrides={}
Item2Vec: training model
  baskets: 788,724
  product tokens: 7,511,295; avg/basket: 9.52; median/basket: 6.00
  model path: /Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/current_config_word2vec_product.model
  config: vector_size=64, window=190 (configured=5, full_basket_context=True), min_count=10, negative=10, sample=0.001, epochs=3, sg=1, workers=1, seed=42
Item2Vec epoch 1/3 started


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Item2Vec epoch 1/3 finished in 214.8s; elapsed=217.7s
Item2Vec epoch 2/3 started


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Item2Vec epoch 2/3 finished in 212.0s; elapsed=429.8s
Item2Vec epoch 3/3 started


...

Item2Vec epoch 4/5 finished in 512.2s; elapsed=2,051.1s
Item2Vec epoch 5/5 started


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Item2Vec epoch 5/5 finished in 512.8s; elapsed=2,563.8s
Item2Vec model saved: vocab=37,299, vector_size=100, path=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/dims100_negative15_word2vec_product.model
[05:52:03] Stage 2 product embeddings: wrote artifact | products=37299, path=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/dims100_negative15_product_embeddings.parquet
[05:52:03] Item2Vec training sandbox: finished | elapsed_s=11934.1000
[05:52:03] Experiment summary: wrote human-readable summaries | rows=7, csv=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_item2vec_training_summary.csv, markdown=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_item2vec_training_summary.md
[05:52:03] Experiment 


### Item2Vec Training Experiment

- Manifest parquet: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_item2vec_training_manifest.parquet`
- Manifest summary CSV: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_item2vec_training_summary.csv`
- Manifest summary Markdown: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_item2vec_training_summary.md`


experiment_name,trial_name,word2vec_overrides,model_path,product_embeddings_path
str,str,str,str,str
"""item2vec_params_sandbox""","""current_config""","""{}""","""/Users/ljyr/Documents/IE/Capst…","""/Users/ljyr/Documents/IE/Capst…"
"""item2vec_params_sandbox""","""dims100_epochs5""","""{""epochs"": 5, ""vector_size"": 1…","""/Users/ljyr/Documents/IE/Capst…","""/Users/ljyr/Documents/IE/Capst…"
"""item2vec_params_sandbox""","""dims128_epochs5""","""{""epochs"": 5, ""vector_size"": 1…","""/Users/ljyr/Documents/IE/Capst…","""/Users/ljyr/Documents/IE/Capst…"
"""item2vec_params_sandbox""","""dims100_min_count5""","""{""epochs"": 5, ""min_count"": 5, …","""/Users/ljyr/Documents/IE/Capst…","""/Users/ljyr/Documents/IE/Capst…"
"""item2vec_params_sandbox""","""dims100_subsample0005""","""{""epochs"": 5, ""sample"": 0.0005…","""/Users/ljyr/Documents/IE/Capst…","""/Users/ljyr/Documents/IE/Capst…"
"""item2vec_params_sandbox""","""dims100_min_count5_subsample00…","""{""epochs"": 5, ""min_count"": 5, …","""/Users/ljyr/Documents/IE/Capst…","""/Users/ljyr/Documents/IE/Capst…"
"""item2vec_params_sandbox""","""dims100_negative15""","""{""epochs"": 5, ""negative"": 15, …","""/Users/ljyr/Documents/IE/Capst…","""/Users/ljyr/Documents/IE/Capst…"


## Experiment 3: Product Embedding Validation

Optional sandbox for evaluating product embedding tables. It can evaluate the current pipeline embedding table, freshly trained Experiment 2 outputs, or existing experiment files already saved under `outputs/<mode>/experiments/<experiment_name>`.


In [ ]:
# Comparing window size
import polars as pl
from IPython.display import Image, Markdown, display

from src.experiment_sandbox import evaluate_product_embedding_experiments

RUN_PRODUCT_EMBEDDING_VALIDATION_SANDBOX = True
embedding_validation_sandbox_experiment_name = globals().get(
    "item2vec_sandbox_experiment_name",
    "item2vec_embedding_sandbox",
)
embedding_validation_sandbox_sample_size = 300 if CONFIG.mode == "dev" else 500
embedding_validation_sandbox_neighbors = 8

if "item2vec_training_sandbox" in globals():
    embedding_validation_sandbox_paths = item2vec_training_sandbox["embedding_paths"]
else:
    embedding_validation_sandbox_paths = {"current_pipeline": product_embeddings_path}
    experiment_dir = CONFIG.experiments / embedding_validation_sandbox_experiment_name
    for trial in globals().get("item2vec_sandbox_trials", []):
        trial_name = trial["name"]
        candidate_path = experiment_dir / f"{trial_name}_product_embeddings.parquet"
        if candidate_path.exists():
            embedding_validation_sandbox_paths[trial_name] = candidate_path

if RUN_PRODUCT_EMBEDDING_VALIDATION_SANDBOX:
    embedding_validation_sandbox = evaluate_product_embedding_experiments(
        embedding_validation_sandbox_paths,
        experiment_name=embedding_validation_sandbox_experiment_name,
        sample_size=embedding_validation_sandbox_sample_size,
        neighbors=embedding_validation_sandbox_neighbors,
        transactions=transactions,
        cfg=CONFIG,
    )
    display(Markdown(f"""
### Product Embedding Validation Experiment

- Diagnostics parquet: `{embedding_validation_sandbox['diagnostics']['parquet']}`
- Diagnostics summary CSV: `{embedding_validation_sandbox['diagnostics']['summary_csv']}`
- Diagnostics summary Markdown: `{embedding_validation_sandbox['diagnostics']['summary_md']}`
- Best trial: `{embedding_validation_sandbox['best']['trial_name']}`
"""))
    display(
        pl.read_parquet(embedding_validation_sandbox["diagnostics"]["parquet"]).select([
            "embedding_sandbox_rank",
            "trial_name",
            "vector_dims",
            "product_vocab_coverage_pct",
            "same_sector_at_1_pct",
            "same_sector_neighbor_share_pct",
            "mean_neighbor_cosine",
            "embedding_quality_score",
            "selected_in_embedding_sandbox",
            "selection_reason",
        ])
    )
else:
    display(Markdown(
        "Product embedding validation sandbox is self-contained and currently disabled. "
        "Set `RUN_PRODUCT_EMBEDDING_VALIDATION_SANDBOX = True` in this cell to compare embedding tables."
    ))

[02:20:43] Product embedding validation sandbox: evaluating embedding tables | experiment=item2vec_window_sandbox, trials=4
[02:20:57] Product embedding validation sandbox: finished | elapsed_s=14.2000
[02:20:57] Experiment summary: wrote human-readable summaries | rows=4, csv=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_embedding_summary.csv, markdown=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_embedding_summary.md
[02:20:57] Experiment sandbox: wrote diagnostics | rows=4, parquet=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_embedding_diagnostics.parquet, summary_csv=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_embedding_summary.csv, summ


### Product Embedding Validation Experiment

- Diagnostics parquet: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_embedding_diagnostics.parquet`
- Diagnostics summary CSV: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_embedding_summary.csv`
- Diagnostics summary Markdown: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_window_sandbox/item2vec_window_sandbox_embedding_summary.md`
- Best trial: `window5`


embedding_sandbox_rank,trial_name,vector_dims,product_vocab_coverage_pct,same_sector_at_1_pct,same_sector_neighbor_share_pct,mean_neighbor_cosine,embedding_quality_score,selected_in_embedding_sandbox,selection_reason
i64,str,i64,f64,f64,f64,f64,f64,bool,str
1,"""window5""",64,47.812488,75.0,69.125,0.808458,66.963736,true,"""Ranked by product vocabulary c…"
2,"""window10""",64,47.812488,75.0,69.125,0.808458,66.963736,false,"""Ranked by product vocabulary c…"
3,"""window15""",64,47.812488,75.0,69.125,0.808458,66.963736,false,"""Ranked by product vocabulary c…"
4,"""window20""",64,47.812488,75.0,69.125,0.808458,66.963736,false,"""Ranked by product vocabulary c…"


In [5]:
# Comparing Item2Vec training parameters
import polars as pl
from IPython.display import Image, Markdown, display

from src.experiment_sandbox import evaluate_product_embedding_experiments

RUN_PRODUCT_EMBEDDING_VALIDATION_SANDBOX = True
embedding_validation_sandbox_experiment_name = globals().get(
    "item2vec_sandbox_experiment_name",
    "item2vec_embedding_sandbox",
)
embedding_validation_sandbox_sample_size = 300 if CONFIG.mode == "dev" else 500
embedding_validation_sandbox_neighbors = 8

if "item2vec_training_sandbox" in globals():
    embedding_validation_sandbox_paths = item2vec_training_sandbox["embedding_paths"]
else:
    embedding_validation_sandbox_paths = {"current_pipeline": product_embeddings_path}
    experiment_dir = CONFIG.experiments / embedding_validation_sandbox_experiment_name
    for trial in globals().get("item2vec_sandbox_trials", []):
        trial_name = trial["name"]
        candidate_path = experiment_dir / f"{trial_name}_product_embeddings.parquet"
        if candidate_path.exists():
            embedding_validation_sandbox_paths[trial_name] = candidate_path

if RUN_PRODUCT_EMBEDDING_VALIDATION_SANDBOX:
    embedding_validation_sandbox = evaluate_product_embedding_experiments(
        embedding_validation_sandbox_paths,
        experiment_name=embedding_validation_sandbox_experiment_name,
        sample_size=embedding_validation_sandbox_sample_size,
        neighbors=embedding_validation_sandbox_neighbors,
        transactions=transactions,
        cfg=CONFIG,
    )
    display(Markdown(f"""
### Product Embedding Validation Experiment

- Diagnostics parquet: `{embedding_validation_sandbox['diagnostics']['parquet']}`
- Diagnostics summary CSV: `{embedding_validation_sandbox['diagnostics']['summary_csv']}`
- Diagnostics summary Markdown: `{embedding_validation_sandbox['diagnostics']['summary_md']}`
- Best trial: `{embedding_validation_sandbox['best']['trial_name']}`
"""))
    display(
        pl.read_parquet(embedding_validation_sandbox["diagnostics"]["parquet"]).select([
            "embedding_sandbox_rank",
            "trial_name",
            "vector_dims",
            "product_vocab_coverage_pct",
            "same_sector_at_1_pct",
            "same_sector_neighbor_share_pct",
            "mean_neighbor_cosine",
            "embedding_quality_score",
            "selected_in_embedding_sandbox",
            "selection_reason",
        ])
    )
else:
    display(Markdown(
        "Product embedding validation sandbox is self-contained and currently disabled. "
        "Set `RUN_PRODUCT_EMBEDDING_VALIDATION_SANDBOX = True` in this cell to compare embedding tables."
    ))

[10:05:13] Product embedding validation sandbox: evaluating embedding tables | experiment=item2vec_params_sandbox, trials=7
[10:05:40] Product embedding validation sandbox: finished | elapsed_s=26.6000
[10:05:40] Experiment summary: wrote human-readable summaries | rows=7, csv=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_embedding_summary.csv, markdown=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_embedding_summary.md
[10:05:40] Experiment sandbox: wrote diagnostics | rows=7, parquet=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_embedding_diagnostics.parquet, summary_csv=/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_embedding_summary.csv, summ


### Product Embedding Validation Experiment

- Diagnostics parquet: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_embedding_diagnostics.parquet`
- Diagnostics summary CSV: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_embedding_summary.csv`
- Diagnostics summary Markdown: `/Users/ljyr/Documents/IE/Capstone Project/carrefour_capstone/outputs/dev/experiments/item2vec_params_sandbox/item2vec_params_sandbox_embedding_summary.md`
- Best trial: `dims100_subsample0005`


embedding_sandbox_rank,trial_name,vector_dims,product_vocab_coverage_pct,same_sector_at_1_pct,same_sector_neighbor_share_pct,mean_neighbor_cosine,embedding_quality_score,selected_in_embedding_sandbox,selection_reason
i64,str,i64,f64,f64,f64,f64,f64,bool,str
1,"""dims100_subsample0005""",100,47.812488,78.0,71.291667,0.671391,68.665706,true,"""Ranked by product vocabulary c…"
2,"""dims100_epochs5""",100,47.812488,75.0,72.458333,0.672079,68.337363,false,"""Ranked by product vocabulary c…"
3,"""dims128_epochs5""",128,47.812488,74.333333,70.875,0.63824,67.568746,false,"""Ranked by product vocabulary c…"
4,"""dims100_negative15""",100,47.812488,74.333333,69.791667,0.673513,67.303531,false,"""Ranked by product vocabulary c…"
5,"""current_config""",64,47.812488,75.0,69.125,0.808458,66.963736,false,"""Ranked by product vocabulary c…"
6,"""dims100_min_count5""",100,62.055351,66.0,62.458333,0.731385,65.462342,false,"""Ranked by product vocabulary c…"
7,"""dims100_min_count5_subsample00…",100,62.055351,64.666667,62.583333,0.731506,65.1627,false,"""Ranked by product vocabulary c…"


# Item2Vec Experiment Results

## Experiment 2-3A: Window Size

**Hypothesis:** Larger windows capture broader shopping missions and may improve product embedding quality.

**Trials:** window5, window10, window15, window20

**Output folder:** `outputs/dev/experiments/item2vec_window_sandbox/`

**Results:** All trials scored identically across all metrics (vocab coverage: 47.81%, same_sector_at_1: 75.0%, same_sector_neighbor_share: 69.125%, mean_cosine: 0.808, quality_score: 66.96).

**Note:** `full_basket_context: true` in `base.yaml` is a deliberate design choice that treats the entire basket as context, making window size irrelevant. This is appropriate for grocery retail where basket order has no meaning. Window size tuning would only be relevant if this setting is changed.

**Best trial:** window5 (tied — no meaningful difference)

**Promote / iterate / reject:** Reject — no evidence to change default. Window size experiments are only meaningful if `full_basket_context` is set to false.

**YAML changes needed:** None

**Official stages to rerun:** None

---

## Experiment 2-3B: Item2Vec Parameters

**Hypothesis:** Increasing vector dimensions and downweighting very frequent products will improve product embedding coherence and sector structure.

**Trials:** current_config, dims100_epochs5, dims128_epochs5, dims100_min_count5, dims100_subsample0005, dims100_min_count5_subsample0005, dims100_negative15

**Output folder:** `outputs/dev/experiments/item2vec_params_sandbox/`

**Results:**

| Rank | Trial | Quality Score | same_sector_at_1 | same_sector_neighbor_share | mean_cosine |
|---|---|---|---|---|---|
| 1 | dims100_subsample0005 | 68.67 | 78.0% | 71.29% | 0.671 |
| 2 | dims100_epochs5 | 68.34 | 75.0% | 72.46% | 0.672 |
| 3 | dims128_epochs5 | 67.57 | 74.33% | 70.88% | 0.638 |
| 4 | dims100_negative15 | 67.30 | 74.33% | 69.79% | 0.674 |
| 5 | current_config | 66.96 | 75.0% | 69.13% | 0.808 |
| 6 | dims100_min_count5 | 65.46 | 66.0% | 62.46% | 0.731 |
| 7 | dims100_min_count5_subsample0005 | 65.16 | 64.67% | 62.58% | 0.732 |

**Best trial:** dims100_subsample0005

**Why it looked better:** Highest same_sector_at_1 (78%) and overall quality score (68.67). Downweighting very frequent products allows the model to learn more distinctive product relationships rather than being dominated by universal staples.

**What got worse:** mean_neighbor_cosine is lower than current_config (0.671 vs 0.808) — neighbors are slightly less tightly clustered overall, but sector coherence is stronger which matters more for tribe discovery.

**Notable finding:** Lowering min_count to 5 increases vocab coverage (62% vs 47.81%) but hurts sector coherence (same_sector_at_1 drops to 66%). More products get embeddings but they are lower quality. Not worth the trade-off.

**Promote / iterate / reject:** Promote

**YAML changes needed:**
```yaml
word2vec:
  vector_size: 100
  sample: 0.0005
```

**Official stages to rerun:** Stage 2 onwards (product embeddings change upstream of everything)

## Experiment 4: Customer Embedding Aggregation

Optional sandbox for testing how product embeddings are aggregated into customer vectors. Official clustering uses product quantities only. IDF variants are tested here before any promotion to YAML.


In [2]:
import polars as pl
from IPython.display import Image, Markdown, display

from src.data_loader import load_prepared_transactions
from src.experiment_sandbox import run_customer_embedding_experiments

RUN_CUSTOMER_EMBEDDING_SANDBOX = True   
customer_embedding_sandbox_experiment_name = "customer_embedding_sandbox"
customer_embedding_sandbox_sample_size = 5000 if CONFIG.mode == "dev" else 12000
customer_embedding_sandbox_trials = [
    {"name": "quantity_weighted", "weight_strategy": "quantity", "normalize_vectors": False},
    {"name": "quantity_idf_weighted", "weight_strategy": "quantity_idf", "normalize_vectors": False},
    {"name": "equal_weighted", "weight_strategy": "equal", "normalize_vectors": False},
    {"name": "equal_idf_weighted", "weight_strategy": "equal_idf", "normalize_vectors": False},
    {"name": "quantity_weighted_l2", "weight_strategy": "quantity", "normalize_vectors": True},
    {"name": "quantity_idf_weighted_l2", "weight_strategy": "quantity_idf", "normalize_vectors": True},
    {"name": "equal_weighted_l2", "weight_strategy": "equal", "normalize_vectors": True},
]

if "product_embeddings_path" not in globals():
    product_embeddings_path = CONFIG.artifact_path(
        "word2vec",
        "embeddings_output",
        directory=CONFIG.outputs / "embeddings",
    )
    if not product_embeddings_path.exists():
        raise FileNotFoundError(
            "Experiment 4 needs product embeddings. Run Stage 2 first or point "
            f"`product_embeddings_path` to an existing file. Missing: {product_embeddings_path}"
        )

if "transactions" not in globals():
    transactions = load_prepared_transactions(cfg=CONFIG)

if RUN_CUSTOMER_EMBEDDING_SANDBOX:
    customer_embedding_sandbox = run_customer_embedding_experiments(
        product_embeddings_path,
        trials=customer_embedding_sandbox_trials,
        experiment_name=customer_embedding_sandbox_experiment_name,
        sample_size=customer_embedding_sandbox_sample_size,
        transactions=transactions,
        force=True,
        cfg=CONFIG,
    )
    display(Markdown(f"""
### Customer Embedding Experiment Results

- Diagnostics parquet: `{customer_embedding_sandbox['diagnostics']['parquet']}`
- Diagnostics summary CSV: `{customer_embedding_sandbox['diagnostics']['summary_csv']}`
- Diagnostics summary Markdown: `{customer_embedding_sandbox['diagnostics']['summary_md']}`
- Best trial: `{customer_embedding_sandbox['best']['trial_name']}`
"""))
    display(
        pl.read_parquet(customer_embedding_sandbox["diagnostics"]["parquet"]).select([
            "customer_embedding_sandbox_rank",
            "trial_name",
            "weight_strategy",
            "normalize_vectors",
            "vector_dims",
            "finite_pct",
            "zero_vector_pct",
            "effective_dimension",
            "mean_nearest_neighbor_cosine",
            "customer_embedding_quality_score",
            "selected_in_customer_embedding_sandbox",
            "selection_reason",
        ])
    )
else:
    display(Markdown(
        "Customer embedding sandbox is self-contained and currently disabled. "
        "Set `RUN_CUSTOMER_EMBEDDING_SANDBOX = True` in this cell to compare customer-vector aggregation variants."
    ))


[22:59:08] Customer embedding sandbox: building customer-vector trials | experiment=customer_embedding_sandbox, trials=7
[22:59:08] Customer embedding sandbox: starting trial | experiment=customer_embedding_sandbox, trial=quantity_weighted, weight_strategy=quantity, normalize_vectors=False
[22:59:08] Stage 4 customer embeddings: aggregating product vectors | output=C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments\customer_embedding_sandbox\quantity_weighted_customer_embeddings.parquet, weight_strategy=quantity, normalize_vectors=False
[22:59:18] Stage 4 customer embeddings: wrote artifact | customers=43998, path=C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments\customer_embedding_sandbox\quantity_weighted_customer_embeddings.parquet
[22:59:18] Stage 4 customer embeddings: finished | elapsed_s=9.6000
[22:59:23] Customer embedding sandbox: starting trial | experiment=customer_embedding_sandbox, trial=quantity_idf_


### Customer Embedding Experiment Results

- Diagnostics parquet: `C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments\customer_embedding_sandbox\customer_embedding_sandbox_customer_embedding_diagnostics.parquet`
- Diagnostics summary CSV: `C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments\customer_embedding_sandbox\customer_embedding_sandbox_customer_embedding_summary.csv`
- Diagnostics summary Markdown: `C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments\customer_embedding_sandbox\customer_embedding_sandbox_customer_embedding_summary.md`
- Best trial: `quantity_weighted_l2`


customer_embedding_sandbox_rank,trial_name,weight_strategy,normalize_vectors,vector_dims,finite_pct,zero_vector_pct,effective_dimension,mean_nearest_neighbor_cosine,customer_embedding_quality_score,selected_in_customer_embedding_sandbox,selection_reason
i64,str,str,bool,i64,f64,f64,f64,f64,f64,bool,str
1,"""quantity_weighted_l2""","""quantity""",true,100,100.0,0.0,24.679095,0.947837,92.142914,true,"""Ranked by finite-value health,…"
2,"""quantity_idf_weighted_l2""","""quantity_idf""",true,100,100.0,0.0,24.490351,0.940532,92.049766,false,"""Ranked by finite-value health,…"
3,"""equal_weighted_l2""","""equal""",true,100,100.0,0.0,21.515716,0.9567,91.238214,false,"""Ranked by finite-value health,…"
4,"""quantity_weighted""","""quantity""",false,100,100.0,0.0,24.360911,0.947837,90.590955,false,"""Ranked by finite-value health,…"
5,"""quantity_idf_weighted""","""quantity_idf""",false,100,100.0,0.0,24.183318,0.940532,90.384309,false,"""Ranked by finite-value health,…"
6,"""equal_weighted""","""equal""",false,100,100.0,0.0,20.983217,0.9567,89.80836,false,"""Ranked by finite-value health,…"
7,"""equal_idf_weighted""","""equal_idf""",false,100,100.0,0.0,21.057202,0.949738,89.668926,false,"""Ranked by finite-value health,…"


## Experiment 4: Customer Embedding Aggregation

**Hypothesis:** The weighting scheme (equal vs quantity, with/without IDF) and L2 normalization change how cleanly customer vectors form tribes.

**Trials:** quantity_weighted, quantity_idf_weighted, equal_weighted, equal_idf_weighted, quantity_weighted_l2, quantity_idf_weighted_l2, equal_weighted_l2

**Output folder:** `outputs/dev/experiments/customer_embedding_sandbox/`

**Results — intrinsic embedding health (Exp 4):**

| Rank | Trial | quality_score | effective_dim | finite_pct | zero_vec_pct |
|---|---|---|---|---|---|
| 1 | quantity_weighted_l2 | 92.14 | 24.68 | 100% | 0% |
| 2 | quantity_idf_weighted_l2 | 92.05 | 24.49 | 100% | 0% |
| 3 | equal_weighted_l2 | 91.24 | 21.52 | 100% | 0% |
| 4 | quantity_weighted (official) | 90.59 | 24.36 | 100% | 0% |

All seven variants are numerically healthy (finite_pct=100%, zero_vector_pct=0%).

**Results — KMeans clustering probe (Exp 5, embeddings_only):**

| Variant | k | silhouette | cluster_size_cv | quality_score | rank |
|---|---|---|---|---|---|
| quantity_weighted_l2 | 12 | 0.032 | 0.40 | 65.74 | 11 |
| quantity_weighted (official) | 15 | 0.030 | 0.61 | 63.60 | 47 |

Silhouette is near-zero and effectively flat across ALL variants (0.02–0.045): no variant separates customers better in the quick KMeans probe. The ranking is driven by cluster balance (cluster_size_cv).

**Best trial:** quantity_weighted_l2

**Why it looked better:** Quantity weighting is the healthiest scheme (Exp 4) and L2 normalization both raises the intrinsic quality score (90.59 → 92.14) and roughly halves cluster imbalance (cv 0.61 → 0.40) with no loss of separability. Normalizing removes the magnitude effect where high-volume customers dominate distances.

**What got worse:** Nothing material — separability is flat everywhere, so there is no trade-off on that axis.

**Notable finding:** Equal weighting tops the Exp 5 composite, but only on balance in a weak KMeans proxy; its separability is no better, and it contradicts the contract's quantity mandate and the Exp 4 health ranking. IDF variants add no real benefit over their non-IDF counterparts.

**Promote / iterate / reject:** Promote L2 normalization (keep quantity weighting). Reject equal and IDF.

**YAML changes needed:**

Ensure the official Stage 4 call honors this flag (pass `normalize_vectors=CONFIG.get("customer_embeddings.normalize_vectors", False)` if it doesn't already read it from config).

**Official stages to rerun:** Stage 4 (customer embeddings) onwards, with `force=True`.

**Recommended confirmation:** Validate quantity+L2 vs the current official in the real UMAP-HDBSCAN run (Experiment 6 / Stage 6) before locking it in, since the sandbox uses a KMeans proxy.

## Experiment 5: Customer Vector / Feature-Set Probes

Optional sandbox for comparing customer-vector feature variants before the full candidate model suite. It uses quick KMeans probes near the client hypothesis range and reports separability, balance, dimensional health, and finite-value checks.

In [3]:
import polars as pl
from IPython.display import Image, Markdown, display

from src.experiment_sandbox import run_feature_set_experiments
from src.feature_engineering import build_feature_set

RUN_FEATURE_SET_SANDBOX = True
feature_set_sandbox_experiment_name = "customer_feature_set_sandbox"
feature_set_sandbox_sample_size = 5000 if CONFIG.mode == "dev" else 12000
feature_set_sandbox_k_values = [10, 12, 15]

if "feature_sets" not in globals():
    feature_set_outputs = CONFIG.get("feature_sets.outputs", {})
    feature_sets = {
        name: CONFIG.outputs / "features" / filename
        for name, filename in feature_set_outputs.items()
    }
    missing_feature_sets = [path for path in feature_sets.values() if not path.exists()]
    if missing_feature_sets:
        raise FileNotFoundError(
            "Experiment 5 needs the official Stage 5 feature-set parquet files. "
            f"Missing: {missing_feature_sets}. Run Stage 5 first."
        )

feature_set_sandbox_inputs = dict(feature_sets)
if "customer_embedding_sandbox" in globals():
    stage4b_experiment_dir = CONFIG.experiments / customer_embedding_sandbox_experiment_name
    for trial_name, embedding_path in customer_embedding_sandbox["customer_embedding_paths"].items():
        embeddings_only_path = build_feature_set(
            embedding_path,
            variant="embeddings_only",
            output_path=stage4b_experiment_dir / f"{trial_name}_feature_set_embeddings_only.parquet",
            force=False,
            cfg=CONFIG,
        )
        feature_set_sandbox_inputs[f"{trial_name}_embeddings_only"] = embeddings_only_path

        if "behavior_path" in globals():
            behavior_path_for_trial = build_feature_set(
                embedding_path,
                behavior_path=behavior_path,
                variant="embeddings_behavior",
                output_path=stage4b_experiment_dir / f"{trial_name}_feature_set_embeddings_behavior.parquet",
                force=False,
                cfg=CONFIG,
            )
            feature_set_sandbox_inputs[f"{trial_name}_embeddings_behavior"] = behavior_path_for_trial

if RUN_FEATURE_SET_SANDBOX:
    feature_set_sandbox = run_feature_set_experiments(
        feature_set_sandbox_inputs,
        experiment_name=feature_set_sandbox_experiment_name,
        k_values=feature_set_sandbox_k_values,
        sample_size=feature_set_sandbox_sample_size,
        cfg=CONFIG,
    )
    display(Markdown(f"""
### Feature-Set Experiment Results

- Diagnostics parquet: `{feature_set_sandbox['diagnostics']['parquet']}`
- Diagnostics summary CSV: `{feature_set_sandbox['diagnostics']['summary_csv']}`
- Diagnostics summary Markdown: `{feature_set_sandbox['diagnostics']['summary_md']}`
- Best feature set: `{feature_set_sandbox['best']['feature_set_name']}` at k=`{feature_set_sandbox['best']['k']}`
"""))
    display(
        pl.read_parquet(feature_set_sandbox["diagnostics"]["parquet"]).select([
            "feature_set_sandbox_rank",
            "feature_set_name",
            "k",
            "feature_count",
            "effective_dimension",
            "silhouette",
            "davies_bouldin",
            "cluster_size_cv",
            "finite_pct",
            "feature_set_quality_score",
            "selected_in_feature_set_sandbox",
            "selection_reason",
        ])
    )
else:
    display(Markdown(
        "Feature-set sandbox is self-contained and currently disabled. Set `RUN_FEATURE_SET_SANDBOX = True` "
        "in this cell when you want to compare pipeline feature variants. If Experiment 4 ran in this kernel, "
        "its customer-vector trials are included automatically."
    ))


[23:04:21] Stage 5 feature set: building model-ready features | variant=embeddings_only, output=C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments\customer_embedding_sandbox\quantity_weighted_feature_set_embeddings_only.parquet
[23:04:21] Stage 5 feature set: wrote artifact | variant=embeddings_only, rows=43998, path=C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments\customer_embedding_sandbox\quantity_weighted_feature_set_embeddings_only.parquet
[23:04:21] Stage 5 feature set: finished | elapsed_s=0.3000
[23:04:21] Stage 5 feature set: building model-ready features | variant=embeddings_behavior, output=C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments\customer_embedding_sandbox\quantity_weighted_feature_set_embeddings_behavior.parquet
[23:04:22] Stage 5 feature set: wrote artifact | variant=embeddings_behavior, rows=43998, path=C:\Users\vivim\OneDrive\Documentos\Capstone\carref

c:\Users\vivim\miniconda3\envs\carrefour\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


[23:04:29] Feature-set sandbox: fitting quick KMeans | feature_set=embeddings_only, k=12, progress=2/3
[23:04:30] Feature-set sandbox: fitting quick KMeans | feature_set=embeddings_only, k=15, progress=3/3
[23:04:31] Feature-set sandbox: fitting quick KMeans | feature_set=embeddings_behavior, k=10, progress=1/3
[23:04:32] Feature-set sandbox: fitting quick KMeans | feature_set=embeddings_behavior, k=12, progress=2/3
[23:04:32] Feature-set sandbox: fitting quick KMeans | feature_set=embeddings_behavior, k=15, progress=3/3
[23:04:33] Feature-set sandbox: fitting quick KMeans | feature_set=quantity_weighted_embeddings_only, k=10, progress=1/3
[23:04:34] Feature-set sandbox: fitting quick KMeans | feature_set=quantity_weighted_embeddings_only, k=12, progress=2/3
[23:04:35] Feature-set sandbox: fitting quick KMeans | feature_set=quantity_weighted_embeddings_only, k=15, progress=3/3
[23:04:36] Feature-set sandbox: fitting quick KMeans | feature_set=quantity_weighted_embeddings_behavior, k=10


### Feature-Set Experiment Results

- Diagnostics parquet: `C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments\customer_feature_set_sandbox\customer_feature_set_sandbox_diagnostics.parquet`
- Diagnostics summary CSV: `C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments\customer_feature_set_sandbox\customer_feature_set_sandbox_summary.csv`
- Diagnostics summary Markdown: `C:\Users\vivim\OneDrive\Documentos\Capstone\carrefour_capstone\outputs\dev\experiments\customer_feature_set_sandbox\customer_feature_set_sandbox_summary.md`
- Best feature set: `equal_idf_weighted_embeddings_behavior` at k=`10`


feature_set_sandbox_rank,feature_set_name,k,feature_count,effective_dimension,silhouette,davies_bouldin,cluster_size_cv,finite_pct,feature_set_quality_score,selected_in_feature_set_sandbox,selection_reason
i64,str,i64,i64,f64,f64,f64,f64,f64,f64,bool,str
1,"""equal_idf_weighted_embeddings_…",10,112,25.516546,0.040349,3.058257,0.248807,100.0,67.260674,true,"""Ranked by quick KMeans separab…"
2,"""equal_weighted_l2_embeddings_o…",10,100,24.608659,0.045199,3.074274,0.309699,100.0,66.876425,false,"""Ranked by quick KMeans separab…"
3,"""equal_weighted_l2_embeddings_o…",15,100,24.608659,0.029578,3.200429,0.274602,100.0,66.797797,false,"""Ranked by quick KMeans separab…"
4,"""equal_weighted_l2_embeddings_b…",10,112,25.882798,0.044872,3.131423,0.348428,100.0,66.405187,false,"""Ranked by quick KMeans separab…"
5,"""equal_idf_weighted_embeddings_…",10,100,24.10415,0.035244,2.978279,0.338504,100.0,66.289372,false,"""Ranked by quick KMeans separab…"
…,…,…,…,…,…,…,…,…,…,…,…
44,"""equal_idf_weighted_embeddings_…",15,112,25.516546,0.022992,3.162153,0.519574,100.0,64.075656,false,"""Ranked by quick KMeans separab…"
45,"""quantity_idf_weighted_embeddin…",12,100,27.062934,0.038651,2.94561,0.60773,100.0,63.838748,false,"""Ranked by quick KMeans separab…"
46,"""embeddings_only""",15,100,27.19273,0.030099,2.82167,0.60903,100.0,63.597074,false,"""Ranked by quick KMeans separab…"


In [4]:
import polars as pl
df = pl.read_parquet(feature_set_sandbox["diagnostics"]["parquet"])
df.filter(pl.col("feature_set_name").str.contains("quantity")).select(
    ["feature_set_sandbox_rank","feature_set_name","k","silhouette","davies_bouldin","cluster_size_cv","feature_set_quality_score"]
).sort("feature_set_sandbox_rank")

feature_set_sandbox_rank,feature_set_name,k,silhouette,davies_bouldin,cluster_size_cv,feature_set_quality_score
i64,str,i64,f64,f64,f64,f64
7,"""quantity_idf_weighted_l2_embed…",12,0.032974,3.245426,0.344145,66.239058
8,"""quantity_idf_weighted_l2_embed…",10,0.03253,3.38809,0.375348,66.015659
11,"""quantity_weighted_l2_embedding…",12,0.032193,3.072955,0.402797,65.739324
12,"""quantity_idf_weighted_l2_embed…",12,0.04004,3.057381,0.42593,65.716368
14,"""quantity_idf_weighted_l2_embed…",10,0.035494,3.264286,0.415514,65.59465
…,…,…,…,…,…,…
40,"""quantity_weighted_l2_embedding…",15,0.027871,3.050664,0.509738,64.551039
41,"""quantity_idf_weighted_l2_embed…",15,0.035845,3.212465,0.540719,64.352248
45,"""quantity_idf_weighted_embeddin…",12,0.038651,2.94561,0.60773,63.838748


## Experiment 6: Focused UMAP-HDBSCAN Trials

Use this optional cell when you want to test UMAP-HDBSCAN settings without rerunning the full Stage 6 candidate suite. It reuses the existing Stage 5 feature set and writes all experiment artifacts directly under `outputs/<mode>/experiments/<experiment_name>`. Trial names describe density assumptions, not target cluster counts.


In [ ]:
import polars as pl
from IPython.display import Image, Markdown, display

from src.model_selection import run_umap_hdbscan_experiments
from src.visualization import plot_stage6_model_diagnostics

RUN_UMAP_HDBSCAN_SANDBOX = False

selection_feature_set = globals().get(
    "selection_feature_set",
    CONFIG.get("modeling.feature_set_for_selection", "embeddings_only"),
)

if "feature_sets" not in globals():
    feature_set_outputs = CONFIG.get("feature_sets.outputs", {})
    feature_sets = {
        name: CONFIG.outputs / "features" / filename
        for name, filename in feature_set_outputs.items()
    }
    missing_feature_sets = [path for path in feature_sets.values() if not path.exists()]
    if missing_feature_sets:
        raise FileNotFoundError(
            "Experiment 6 needs the official Stage 5 feature-set parquet files. "
            f"Missing: {missing_feature_sets}. Run Stage 5 first."
        )

manual_umap_hdbscan_trials = [
    {
        "name": "local_leaf",
        "umap": {
            "n_components": 20,
            "n_neighbors": 30,
            "min_dist": 0.0,
            "metric": "cosine",
        },
        "hdbscan": {
            "min_cluster_size": 300,
            "min_samples": 1,
            "cluster_selection_method": "leaf",
        },
    },
    {
        "name": "balanced_eom",
        "umap": {
            "n_components": 20,
            "n_neighbors": 50,
            "min_dist": 0.0,
            "metric": "cosine",
        },
        "hdbscan": {
            "min_cluster_size": 500,
            "min_samples": 1,
            "cluster_selection_method": "eom",
        },
    },
    {
        "name": "broad_eom",
        "umap": {
            "n_components": 15,
            "n_neighbors": 75,
            "min_dist": 0.0,
            "metric": "cosine",
        },
        "hdbscan": {
            "min_cluster_size": 750,
            "min_samples": 1,
            "cluster_selection_method": "eom",
        },
    },
]

experiment_name = "umap_hdbscan_density_trials"

if RUN_UMAP_HDBSCAN_SANDBOX:
    experiment_suite = run_umap_hdbscan_experiments(
        feature_sets[selection_feature_set],
        trials=manual_umap_hdbscan_trials,
        experiment_name=experiment_name,
        force=True,
        cfg=CONFIG,
    )
    experiment_figure = plot_stage6_model_diagnostics(
        experiment_suite["diagnostics"]["parquet"],
        output_path=CONFIG.figures / f"stage6b_{experiment_name}_diagnostics.png",
        cfg=CONFIG,
    )

    display(Markdown(f"""
### Focused UMAP-HDBSCAN Experiment Results

- Diagnostics parquet: `{experiment_suite['diagnostics']['parquet']}`
- Diagnostics summary CSV: `{experiment_suite['diagnostics']['summary_csv']}`
- Diagnostics summary Markdown: `{experiment_suite['diagnostics']['summary_md']}`
- Diagnostics figure: `{experiment_figure}`
"""))
    display(Image(filename=str(experiment_figure)))

    display(pl.read_parquet(experiment_suite["diagnostics"]["parquet"]).select([
        "stage6_rank",
        "model_name",
        "trial_name",
        "model_variant",
        "cluster_count",
        "coverage_adjusted_silhouette",
        "silhouette",
        "davies_bouldin",
        "noise_pct",
        "passes_quality_gate",
    ]))
else:
    display(Markdown(
        "Focused UMAP-HDBSCAN sandbox is self-contained and currently disabled. "
        "Set `RUN_UMAP_HDBSCAN_SANDBOX = True` in this cell to run these trials."
    ))


## Experiment 7: Low-Dimensional UMAP Target Sweep

Use this section after Experiment 6 when high-dimensional UMAP-HDBSCAN finds either too few broad clusters or too much noise. The first block keeps the density-based approach but lowers the UMAP clustering space to 5-10 dimensions. The second block keeps UMAP as the dimensionality reduction step and tests fixed `k=10..15` clusters on the reduced coordinates.

In [ ]:
import numpy as np
import polars as pl
from IPython.display import Image, Markdown, display
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

from src.dimensionality import build_umap_representation
from src.evaluation import evaluate_labels, quality_gate_result
from src.experiment_reporting import write_summary_artifacts
from src.model_selection import run_umap_hdbscan_experiments
from src.utils import deterministic_sample_indices, frame_to_numpy, numeric_feature_columns
from src.visualization import (
    plot_experiment8_shortlist_cluster_sizes,
    plot_experiment8_shortlist_dashboard,
    plot_experiment8_shortlist_theme_heatmap,
    plot_experiment8_shortlist_umap_grid,
    plot_stage6_model_diagnostics,
)

RUN_LOW_DIM_UMAP_HDBSCAN_SANDBOX = False
RUN_UMAP_FIXED_K_SANDBOX = False

selection_feature_set = globals().get(
    'selection_feature_set',
    CONFIG.get('modeling.feature_set_for_selection', 'embeddings_only'),
)

if 'feature_sets' not in globals():
    feature_set_outputs = CONFIG.get('feature_sets.outputs', {})
    feature_sets = {
        name: CONFIG.outputs / 'features' / filename
        for name, filename in feature_set_outputs.items()
    }

feature_path = feature_sets[selection_feature_set]
if not feature_path.exists():
    raise FileNotFoundError(
        'Experiment 7 needs the official Stage 5 feature-set parquet file. '
        f'Missing: {feature_path}. Run Notebook 03 through Stage 5 first.'
    )

low_dim_umap_hdbscan_trials = [
    {
        'name': 'u5_n15_leaf_mcs300',
        'umap': {'n_components': 5, 'n_neighbors': 15, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 300, 'min_samples': 1, 'cluster_selection_method': 'leaf'},
    },
    {
        'name': 'u5_n30_leaf_mcs400',
        'umap': {'n_components': 5, 'n_neighbors': 30, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 400, 'min_samples': 1, 'cluster_selection_method': 'leaf'},
    },
    {
        'name': 'u8_n15_leaf_mcs350',
        'umap': {'n_components': 8, 'n_neighbors': 15, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 350, 'min_samples': 1, 'cluster_selection_method': 'leaf'},
    },
    {
        'name': 'u8_n30_leaf_mcs450',
        'umap': {'n_components': 8, 'n_neighbors': 30, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 450, 'min_samples': 1, 'cluster_selection_method': 'leaf'},
    },
    {
        'name': 'u10_n30_eom_mcs300',
        'umap': {'n_components': 10, 'n_neighbors': 30, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 300, 'min_samples': 1, 'cluster_selection_method': 'eom'},
    },
    {
        'name': 'u10_n50_leaf_mcs500',
        'umap': {'n_components': 10, 'n_neighbors': 50, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 500, 'min_samples': 1, 'cluster_selection_method': 'leaf'},
    },
]

fixed_k_umap_trials = [
    {'name': 'u5_n15', 'umap': {'n_components': 5, 'n_neighbors': 15, 'min_dist': 0.0, 'metric': 'cosine'}},
    {'name': 'u5_n30', 'umap': {'n_components': 5, 'n_neighbors': 30, 'min_dist': 0.0, 'metric': 'cosine'}},
    {'name': 'u8_n30', 'umap': {'n_components': 8, 'n_neighbors': 30, 'min_dist': 0.0, 'metric': 'cosine'}},
    {'name': 'u10_n50', 'umap': {'n_components': 10, 'n_neighbors': 50, 'min_dist': 0.0, 'metric': 'cosine'}},
]
fixed_k_values = list(range(10, 16))


def _sandbox_slug(value):
    return ''.join(char if char.isalnum() else '_' for char in str(value).strip().lower()).strip('_') or 'trial'


def _rank_stage6_like_rows(rows):
    def rank_key(row):
        score = row.get('coverage_adjusted_silhouette')
        if score is None:
            score = row.get('silhouette')
        return (
            0 if row.get('passes_quality_gate') else 1,
            -float(score) if score is not None else 999.0,
            float(row.get('davies_bouldin')) if row.get('davies_bouldin') is not None else 999.0,
            float(row.get('cluster_size_cv')) if row.get('cluster_size_cv') is not None else 999.0,
            float(row.get('noise_pct')) if row.get('noise_pct') is not None else 999.0,
            str(row.get('candidate_id')),
        )

    ranked = sorted(rows, key=rank_key)
    for rank, row in enumerate(ranked, start=1):
        row['stage6_rank'] = rank
        row['selected_within_family'] = rank == 1 and bool(row.get('passes_quality_gate'))
        row['stage6_score'] = row.get('coverage_adjusted_silhouette')
    return ranked


def run_umap_fixed_k_sandbox(feature_path, trials, k_values, experiment_name, force=True, cfg=CONFIG):
    experiment_slug = _sandbox_slug(experiment_name)
    experiment_dir = cfg.experiments / experiment_slug
    experiment_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    for trial in trials:
        trial_name = _sandbox_slug(trial['name'])
        umap_overrides = dict(trial.get('umap', {}))
        umap_path = build_umap_representation(
            feature_path,
            output_path=experiment_dir / f'{trial_name}_umap.parquet',
            umap_overrides=umap_overrides,
            force=force,
            cfg=cfg,
        )

        df = pl.read_parquet(umap_path).sort('cliente')
        feature_cols = numeric_feature_columns(df)
        X = frame_to_numpy(df, feature_cols)
        X_model = StandardScaler().fit_transform(X).astype(np.float32)
        fit_idx = deterministic_sample_indices(X_model.shape[0], cfg.get('modeling.fit_sample_size'), cfg.random_seed)
        clientes = df['cliente'].to_list()

        for k in k_values:
            model = MiniBatchKMeans(
                n_clusters=int(k),
                random_state=cfg.random_seed,
                batch_size=int(cfg.get('kmeans.batch_size', 4096)),
                n_init=int(cfg.get('kmeans.n_init', 10)),
            )
            model.fit(X_model[fit_idx])
            labels = model.predict(X_model).astype(np.int32)
            metrics = evaluate_labels(X_model, labels, cfg=cfg)
            passes_gate, gate_reason = quality_gate_result(metrics, cfg=cfg)
            variant = f'{trial_name}_k{k}'
            model_name = f'experiment_{experiment_slug}_{trial_name}'
            candidate_id = f'{model_name}::{variant}'
            assignment_path = experiment_dir / f'cluster_assignments_{variant}.parquet'

            pl.DataFrame(
                {
                    'cliente': clientes,
                    'tribe_id': labels,
                    'model_name': [model_name] * len(clientes),
                    'model_variant': [variant] * len(clientes),
                    'assignment_probability': [None] * len(clientes),
                    'assignment_confidence_type': [None] * len(clientes),
                    'assignment_source': ['umap_kmeans_predict'] * len(clientes),
                }
            ).write_parquet(assignment_path)

            rows.append(
                {
                    'model': 'UMAP Fixed-k Experiment',
                    'model_id': model_name,
                    'model_name': model_name,
                    'algorithm_name': 'UMAP_MiniBatchKMeans',
                    'trial_name': trial_name,
                    'model_variant': variant,
                    'candidate_id': candidate_id,
                    'feature_space': f'umap_customer_embeddings_{experiment_slug}_{trial_name}',
                    'umap_n_components': int(umap_overrides.get('n_components', cfg.get('umap.n_components'))),
                    'umap_n_neighbors': int(umap_overrides.get('n_neighbors', cfg.get('umap.n_neighbors'))),
                    'umap_min_dist': float(umap_overrides.get('min_dist', cfg.get('umap.min_dist', 0.0))),
                    'k': int(k),
                    **metrics,
                    'passes_quality_gate': passes_gate,
                    'quality_gate_reason': gate_reason,
                    'assignment_path': str(assignment_path),
                    'source_result_path': None,
                }
            )

    ranked = _rank_stage6_like_rows(rows)
    diagnostics_path = experiment_dir / f'{experiment_slug}_diagnostics.parquet'
    diagnostics = pl.from_dicts(ranked, infer_schema_length=None)
    diagnostics.write_parquet(diagnostics_path)
    summaries = write_summary_artifacts(
        diagnostics,
        output_base=diagnostics_path,
        title='UMAP Fixed-k Target Sweep Diagnostics',
        priority_columns=[
            'stage6_rank',
            'passes_quality_gate',
            'trial_name',
            'model_variant',
            'cluster_count',
            'stage6_score',
            'coverage_adjusted_silhouette',
            'silhouette',
            'davies_bouldin',
            'noise_pct',
            'cluster_size_cv',
        ],
        cfg=cfg,
    )
    return {'parquet': diagnostics_path, **summaries}


def _display_stage6_table(diagnostics_path):
    return (
        pl.read_parquet(diagnostics_path)
        .with_columns(
            pl.col('cluster_count').is_between(10, 15).alias('in_10_15_range')
        )
        .select(
            [
                'stage6_rank',
                'trial_name',
                'model_variant',
                'cluster_count',
                'in_10_15_range',
                'coverage_adjusted_silhouette',
                'silhouette',
                'davies_bouldin',
                'noise_pct',
                'cluster_size_cv',
                'passes_quality_gate',
            ]
        )
    )


if RUN_LOW_DIM_UMAP_HDBSCAN_SANDBOX:
    low_dim_experiment_name = 'umap_low_dim_hdbscan_target_sweep'
    low_dim_suite = run_umap_hdbscan_experiments(
        feature_path,
        trials=low_dim_umap_hdbscan_trials,
        experiment_name=low_dim_experiment_name,
        force=True,
        cfg=CONFIG,
    )
    low_dim_figure = plot_stage6_model_diagnostics(
        low_dim_suite['diagnostics']['parquet'],
        output_path=CONFIG.figures / f'stage6b_{low_dim_experiment_name}_diagnostics.png',
        cfg=CONFIG,
    )
    display(Markdown('\n'.join([
        '### Low-Dimensional UMAP-HDBSCAN Results',
        '',
        f'- Diagnostics parquet: `{low_dim_suite["diagnostics"]["parquet"]}`',
        f'- Summary CSV: `{low_dim_suite["diagnostics"]["summary_csv"]}`',
        f'- Summary Markdown: `{low_dim_suite["diagnostics"]["summary_md"]}`',
        f'- Diagnostics figure: `{low_dim_figure}`',
    ])))
    display(Image(filename=str(low_dim_figure)))
    display(_display_stage6_table(low_dim_suite['diagnostics']['parquet']))
else:
    display(Markdown(
        'Low-dimensional UMAP-HDBSCAN target sweep is disabled. '
        'Set `RUN_LOW_DIM_UMAP_HDBSCAN_SANDBOX = True` to run it.'
    ))


if RUN_UMAP_FIXED_K_SANDBOX:
    fixed_k_experiment_name = 'umap_fixed_k_target_sweep'
    fixed_k_diagnostics = run_umap_fixed_k_sandbox(
        feature_path,
        trials=fixed_k_umap_trials,
        k_values=fixed_k_values,
        experiment_name=fixed_k_experiment_name,
        force=True,
        cfg=CONFIG,
    )
    fixed_k_figure = plot_stage6_model_diagnostics(
        fixed_k_diagnostics['parquet'],
        output_path=CONFIG.figures / f'stage6b_{fixed_k_experiment_name}_diagnostics.png',
        cfg=CONFIG,
    )
    display(Markdown('\n'.join([
        '### UMAP Fixed-k Results',
        '',
        f'- Diagnostics parquet: `{fixed_k_diagnostics["parquet"]}`',
        f'- Summary CSV: `{fixed_k_diagnostics["summary_csv"]}`',
        f'- Summary Markdown: `{fixed_k_diagnostics["summary_md"]}`',
        f'- Diagnostics figure: `{fixed_k_figure}`',
    ])))
    display(Image(filename=str(fixed_k_figure)))
    display(_display_stage6_table(fixed_k_diagnostics['parquet']))
else:
    display(Markdown(
        'UMAP fixed-k target sweep is disabled. '
        'Set `RUN_UMAP_FIXED_K_SANDBOX = True` to run the k=10..15 fallback.'
    ))


## Experiment 7B: UMAP-HDBSCAN Noise-Reduction Follow-Up

Run this after the low-dimensional sweep if the best target-range candidate still has slightly too much noise. These trials search around the promising `u8_n15_leaf_mcs350` result and add lower-`min_cluster_size` EOM variants, which may recover more coverage without collapsing all the way to 2-3 broad clusters.

In [ ]:
import polars as pl
from IPython.display import Image, Markdown, display

from src.model_selection import run_umap_hdbscan_experiments
from src.visualization import plot_stage6_model_diagnostics

RUN_UMAP_HDBSCAN_NOISE_REDUCTION_SWEEP = False

selection_feature_set = globals().get(
    'selection_feature_set',
    CONFIG.get('modeling.feature_set_for_selection', 'embeddings_only'),
)

if 'feature_sets' not in globals():
    feature_set_outputs = CONFIG.get('feature_sets.outputs', {})
    feature_sets = {
        name: CONFIG.outputs / 'features' / filename
        for name, filename in feature_set_outputs.items()
    }

feature_path = feature_sets[selection_feature_set]
if not feature_path.exists():
    raise FileNotFoundError(
        'Experiment 7B needs the official Stage 5 feature-set parquet file. '
        f'Missing: {feature_path}. Run Notebook 03 through Stage 5 first.'
    )

noise_reduction_umap_hdbscan_trials = [
    # Leaf variants around the best 10-15-cluster result from Experiment 7.
    {
        'name': 'u8_n20_leaf_mcs325',
        'umap': {'n_components': 8, 'n_neighbors': 20, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 325, 'min_samples': 1, 'cluster_selection_method': 'leaf'},
    },
    {
        'name': 'u8_n25_leaf_mcs350',
        'umap': {'n_components': 8, 'n_neighbors': 25, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 350, 'min_samples': 1, 'cluster_selection_method': 'leaf'},
    },
    {
        'name': 'u8_n30_leaf_mcs350',
        'umap': {'n_components': 8, 'n_neighbors': 30, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 350, 'min_samples': 1, 'cluster_selection_method': 'leaf'},
    },
    {
        'name': 'u10_n40_leaf_mcs350',
        'umap': {'n_components': 10, 'n_neighbors': 40, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 350, 'min_samples': 1, 'cluster_selection_method': 'leaf'},
    },
    {
        'name': 'u10_n50_leaf_mcs350',
        'umap': {'n_components': 10, 'n_neighbors': 50, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 350, 'min_samples': 1, 'cluster_selection_method': 'leaf'},
    },
    {
        'name': 'u10_n50_leaf_mcs400',
        'umap': {'n_components': 10, 'n_neighbors': 50, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 400, 'min_samples': 1, 'cluster_selection_method': 'leaf'},
    },
    # EOM variants: lower min_cluster_size than the 3-cluster low-noise run.
    {
        'name': 'u8_n20_eom_mcs150',
        'umap': {'n_components': 8, 'n_neighbors': 20, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 150, 'min_samples': 1, 'cluster_selection_method': 'eom'},
    },
    {
        'name': 'u10_n30_eom_mcs150',
        'umap': {'n_components': 10, 'n_neighbors': 30, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 150, 'min_samples': 1, 'cluster_selection_method': 'eom'},
    },
    {
        'name': 'u10_n50_eom_mcs200',
        'umap': {'n_components': 10, 'n_neighbors': 50, 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': 200, 'min_samples': 1, 'cluster_selection_method': 'eom'},
    },
]


def _target_sweep_table(diagnostics_path):
    return (
        pl.read_parquet(diagnostics_path)
        .with_columns(
            pl.col('cluster_count').is_between(10, 15).alias('in_10_15_range')
        )
        .select([
            'stage6_rank',
            'trial_name',
            'model_variant',
            'cluster_count',
            'in_10_15_range',
            'coverage_adjusted_silhouette',
            'silhouette',
            'davies_bouldin',
            'noise_pct',
            'cluster_size_cv',
            'passes_quality_gate',
        ])
    )


if RUN_UMAP_HDBSCAN_NOISE_REDUCTION_SWEEP:
    noise_reduction_experiment_name = 'umap_hdbscan_noise_reduction_sweep'
    noise_reduction_suite = run_umap_hdbscan_experiments(
        feature_path,
        trials=noise_reduction_umap_hdbscan_trials,
        experiment_name=noise_reduction_experiment_name,
        force=True,
        cfg=CONFIG,
    )
    noise_reduction_figure = plot_stage6_model_diagnostics(
        noise_reduction_suite['diagnostics']['parquet'],
        output_path=CONFIG.figures / f'stage6b_{noise_reduction_experiment_name}_diagnostics.png',
        cfg=CONFIG,
    )
    display(Markdown('\n'.join([
        '### UMAP-HDBSCAN Noise-Reduction Sweep Results',
        '',
        f'- Diagnostics parquet: `{noise_reduction_suite["diagnostics"]["parquet"]}`',
        f'- Summary CSV: `{noise_reduction_suite["diagnostics"]["summary_csv"]}`',
        f'- Summary Markdown: `{noise_reduction_suite["diagnostics"]["summary_md"]}`',
        f'- Diagnostics figure: `{noise_reduction_figure}`',
    ])))
    display(Image(filename=str(noise_reduction_figure)))
    display(_target_sweep_table(noise_reduction_suite['diagnostics']['parquet']))
else:
    display(Markdown(
        'UMAP-HDBSCAN noise-reduction sweep is disabled. '
        'Set `RUN_UMAP_HDBSCAN_NOISE_REDUCTION_SWEEP = True` to run this follow-up.'
    ))


## Experiment 8: Consolidated Dimensionality Reduction Search

This section consolidates Experiments 6, 7, and 7B into one larger dimensionality-reduction search. It also includes a focused hard UMAP-HDBSCAN noise-reduction grid centered on the current official failure mode: target-range LEAF clusters with high retained noise. Run the focused grid first, then promote exactly one recipe back into `configs/base.yaml` if the evidence holds.

In [ ]:
import polars as pl
import importlib
from IPython.display import Image, Markdown, display

import src.dimensionality_experiments as dimensionality_experiments

importlib.reload(dimensionality_experiments)

from src.dimensionality_experiments import (
    build_combined_dimensionality_diagnostics,
    build_dimensionality_family_comparison,
    consolidated_umap_hdbscan_trials,
    default_dimensionality_profile_shortlist,
    focused_umap_hdbscan_noise_reduction_trials,
    profile_dimensionality_shortlist,
    ranked_dimensionality_profile_shortlist,
    run_consolidated_single_stage_experiment,
    run_noise_reduction_grid_experiment,
    run_soft_noise_assignment_experiments,
    run_two_stage_hdbscan_experiments,
    two_stage_hdbscan_trials,
)
from src.visualization import plot_stage6_model_diagnostics

# Run these in chunks. The single-stage search is the longest block.
RUN_FOCUSED_NOISE_REDUCTION_GRID = False
RUN_MASTER_SINGLE_STAGE_UMAP_HDBSCAN = False
RUN_MASTER_TWO_STAGE_HDBSCAN = False
RUN_MASTER_SOFT_ASSIGNMENT = False
USE_EXISTING_EXPERIMENT8_ARTIFACTS = False
BUILD_MASTER_COMBINED_DIAGNOSTICS = True
PROFILE_MASTER_SHORTLIST = True

# Use a small integer for a quick smoke run, then set to None for the full focused grid.
FOCUSED_NOISE_REDUCTION_GRID_LIMIT = None

# Use a small integer for a quick smoke run, then set to None for the full search.
MASTER_SINGLE_STAGE_TRIAL_LIMIT = None

focused_noise_grid_experiment_name = 'umap_hdbscan_noise_reduction_grid'
focused_noise_grid_experiment_dir = CONFIG.experiments / focused_noise_grid_experiment_name
master_experiment_name = 'umap_dimensionality_reduction_master_search'
master_experiment_dir = CONFIG.experiments / master_experiment_name

selection_feature_set = globals().get(
    'selection_feature_set',
    CONFIG.get('modeling.feature_set_for_selection', 'embeddings_only'),
)

if 'feature_sets' not in globals():
    feature_set_outputs = CONFIG.get('feature_sets.outputs', {})
    feature_sets = {
        name: CONFIG.outputs / 'features' / filename
        for name, filename in feature_set_outputs.items()
    }

feature_path = feature_sets[selection_feature_set]
if not feature_path.exists():
    raise FileNotFoundError(
        'Experiment 8 needs the official Stage 5 feature-set parquet file. '
        f'Missing: {feature_path}. Run Notebook 03 through Stage 5 first.'
    )

focused_noise_grid_diagnostics_path = focused_noise_grid_experiment_dir / f'{focused_noise_grid_experiment_name}_diagnostics.parquet'
single_stage_diagnostics_path = master_experiment_dir / f'{master_experiment_name}_diagnostics.parquet'
two_stage_diagnostics_path = master_experiment_dir / f'{master_experiment_name}_two_stage_diagnostics.parquet'
single_stage_soft_diagnostics_path = master_experiment_dir / f'{master_experiment_name}_soft_assignment_single_stage_diagnostics.parquet'
two_stage_soft_diagnostics_path = master_experiment_dir / f'{master_experiment_name}_soft_assignment_two_stage_diagnostics.parquet'
combined_diagnostics_path = master_experiment_dir / f'{master_experiment_name}_combined_diagnostics.parquet'
family_comparison_path = master_experiment_dir / f'{master_experiment_name}_family_comparison.parquet'

display(Markdown('\n'.join([
    '### Experiment 8 Search Plan',
    '',
    f'- Focused hard-noise grid trials: `{len(focused_umap_hdbscan_noise_reduction_trials())}`',
    f'- Single-stage UMAP-HDBSCAN trials: `{len(consolidated_umap_hdbscan_trials())}`',
    f'- Two-stage EOM-to-LEAF trials: `{len(two_stage_hdbscan_trials())}`',
    f'- Profile shortlist candidates: `{len(default_dimensionality_profile_shortlist())}`',
    f'- Existing artifacts allowed: `{USE_EXISTING_EXPERIMENT8_ARTIFACTS}`',
    f'- Output folder: `{master_experiment_dir}`',
])))

diagnostics_to_combine = []

if RUN_FOCUSED_NOISE_REDUCTION_GRID:
    focused_noise_grid_suite = run_noise_reduction_grid_experiment(
        feature_path,
        experiment_name=focused_noise_grid_experiment_name,
        limit=FOCUSED_NOISE_REDUCTION_GRID_LIMIT,
        force=True,
        cfg=CONFIG,
    )
    diagnostics_to_combine.append(focused_noise_grid_suite['diagnostics']['parquet'])
    display(Markdown(f"Focused noise-reduction diagnostics: `{focused_noise_grid_suite['diagnostics']['parquet']}`"))
elif USE_EXISTING_EXPERIMENT8_ARTIFACTS and focused_noise_grid_diagnostics_path.exists():
    diagnostics_to_combine.append(focused_noise_grid_diagnostics_path)

if RUN_MASTER_SINGLE_STAGE_UMAP_HDBSCAN:
    master_single_stage_suite = run_consolidated_single_stage_experiment(
        feature_path,
        experiment_name=master_experiment_name,
        limit=MASTER_SINGLE_STAGE_TRIAL_LIMIT,
        force=True,
        cfg=CONFIG,
    )
    diagnostics_to_combine.append(master_single_stage_suite['diagnostics']['parquet'])
    display(Markdown(f"Single-stage diagnostics: `{master_single_stage_suite['diagnostics']['parquet']}`"))
elif USE_EXISTING_EXPERIMENT8_ARTIFACTS and single_stage_diagnostics_path.exists():
    diagnostics_to_combine.append(single_stage_diagnostics_path)

if RUN_MASTER_TWO_STAGE_HDBSCAN:
    master_two_stage_suite = run_two_stage_hdbscan_experiments(
        feature_path,
        experiment_name=master_experiment_name,
        force=True,
        cfg=CONFIG,
    )
    diagnostics_to_combine.append(master_two_stage_suite['diagnostics']['parquet'])
    display(Markdown(f"Two-stage diagnostics: `{master_two_stage_suite['diagnostics']['parquet']}`"))
elif USE_EXISTING_EXPERIMENT8_ARTIFACTS and two_stage_diagnostics_path.exists():
    diagnostics_to_combine.append(two_stage_diagnostics_path)

if RUN_MASTER_SOFT_ASSIGNMENT:
    soft_sources = []
    if single_stage_diagnostics_path.exists():
        soft_sources.append(('soft_assignment_single_stage', single_stage_diagnostics_path))
    if two_stage_diagnostics_path.exists():
        soft_sources.append(('soft_assignment_two_stage', two_stage_diagnostics_path))

    if not soft_sources:
        display(Markdown('Soft assignment skipped because no base diagnostics exist yet.'))
    for soft_suffix, soft_source_path in soft_sources:
        master_soft_diagnostics = run_soft_noise_assignment_experiments(
            soft_source_path,
            experiment_name=master_experiment_name,
            output_suffix=soft_suffix,
            strategies=('q95', 'all'),
            max_candidates=12,
            cfg=CONFIG,
        )
        diagnostics_to_combine.append(master_soft_diagnostics['parquet'])
        display(Markdown(f"Soft-assignment diagnostics: `{master_soft_diagnostics['parquet']}`"))
elif USE_EXISTING_EXPERIMENT8_ARTIFACTS:
    if single_stage_soft_diagnostics_path.exists():
        diagnostics_to_combine.append(single_stage_soft_diagnostics_path)
    if two_stage_soft_diagnostics_path.exists():
        diagnostics_to_combine.append(two_stage_soft_diagnostics_path)

if BUILD_MASTER_COMBINED_DIAGNOSTICS and diagnostics_to_combine:
    master_combined_diagnostics = build_combined_dimensionality_diagnostics(
        diagnostics_to_combine,
        output_path=combined_diagnostics_path,
        cfg=CONFIG,
    )
    master_family_comparison = build_dimensionality_family_comparison(
        master_combined_diagnostics['parquet'],
        output_path=family_comparison_path,
        cfg=CONFIG,
    )
    master_figure = plot_stage6_model_diagnostics(
        master_combined_diagnostics['parquet'],
        output_path=CONFIG.figures / f'stage6b_{master_experiment_name}_combined_diagnostics.png',
        cfg=CONFIG,
    )
    display(Markdown('\n'.join([
        '### Consolidated Dimensionality Reduction Results',
        '',
        f'- Diagnostics parquet: `{master_combined_diagnostics["parquet"]}`',
        f'- Summary CSV: `{master_combined_diagnostics["summary_csv"]}`',
        f'- Summary Markdown: `{master_combined_diagnostics["summary_md"]}`',
        f'- Family comparison CSV: `{master_family_comparison["summary_csv"]}`',
        f'- Family comparison Markdown: `{master_family_comparison["summary_md"]}`',
        f'- Diagnostics figure: `{master_figure}`',
    ])))
    display(Image(filename=str(master_figure)))
    display(Markdown('### Side-by-side experiment family comparison'))
    display(
        pl.read_parquet(master_family_comparison['parquet'])
        .select([
            'experiment_family_label',
            'status',
            'candidate_count',
            'target_range_candidate_count',
            'passing_candidate_count',
            'best_overall_trial',
            'best_overall_clusters',
            'best_overall_score',
            'best_overall_noise_pct',
            'best_target_trial',
            'best_target_clusters',
            'best_target_score',
            'best_target_noise_pct',
            'best_target_passes_gate',
            'comparison_note',
        ])
    )
    display(Markdown('### Ranked candidate-level diagnostics'))
    display(
        pl.read_parquet(master_combined_diagnostics['parquet'])
        .with_columns(pl.col('cluster_count').is_between(10, 15).alias('in_10_15_range'))
        .select([
            'stage6_rank',
            'algorithm_name',
            'trial_name',
            'model_variant',
            'cluster_count',
            'in_10_15_range',
            'coverage_adjusted_silhouette',
            'silhouette',
            'davies_bouldin',
            'noise_pct',
            'cluster_size_cv',
            'passes_quality_gate',
        ])
        .head(30)
    )
    if PROFILE_MASTER_SHORTLIST:
        ranked_profile_shortlist = ranked_dimensionality_profile_shortlist(
            master_combined_diagnostics['parquet'],
            max_candidates=5,
            cfg=CONFIG,
        )
        master_profile_shortlist = profile_dimensionality_shortlist(
            master_combined_diagnostics['parquet'],
            experiment_name=master_experiment_name,
            shortlist=ranked_profile_shortlist,
            force=False,
            cfg=CONFIG,
        )
        display(Markdown('\n'.join([
            '### Experiment 8 shortlist profile comparison',
            '',
            f'- Profile summary CSV: `{master_profile_shortlist["summary_csv"]}`',
            f'- Profile summary Markdown: `{master_profile_shortlist["summary_md"]}`',
            f'- Profile details Markdown: `{master_profile_shortlist["details_md"]}`',
        ])))
        display(
            pl.read_parquet(master_profile_shortlist['parquet'])
            .select([
                'shortlist_label',
                'shortlist_role',
                'cluster_count',
                'coverage_pct',
                'noise_pct',
                'silhouette',
                'davies_bouldin',
                'clusters_with_product_lift',
                'clusters_with_sector_lift',
                'avg_max_product_lift',
                'median_cluster_customers',
                'max_population_share_pct',
                'comparison_note',
            ])
        )
        dashboard_figure = plot_experiment8_shortlist_dashboard(
            master_profile_shortlist['parquet'],
            cfg=CONFIG,
        )
        cluster_size_figure = plot_experiment8_shortlist_cluster_sizes(
            master_profile_shortlist['parquet'],
            cfg=CONFIG,
        )
        theme_heatmap_figure = plot_experiment8_shortlist_theme_heatmap(
            master_profile_shortlist['parquet'],
            cfg=CONFIG,
        )
        projection_grid_figure = plot_experiment8_shortlist_umap_grid(
            feature_path,
            master_profile_shortlist['parquet'],
            max_candidates=4,
            max_points=12000,
            projection_method='pca',
            cfg=CONFIG,
        )
        display(Markdown('\n'.join([
            '### Experiment 8 visual review',
            '',
            f'- Decision dashboard: `{dashboard_figure}`',
            f'- Cluster sizes: `{cluster_size_figure}`',
            f'- Theme heatmap: `{theme_heatmap_figure}`',
            f'- Shared PCA projection: `{projection_grid_figure}`',
        ])))
        for figure_path in [dashboard_figure, cluster_size_figure, theme_heatmap_figure, projection_grid_figure]:
            display(Image(filename=str(figure_path)))
else:
    display(Markdown(
        'Experiment 8 is configured but no fresh diagnostics were run or loaded. '
        'Set `RUN_FOCUSED_NOISE_REDUCTION_GRID = True` for the current Stage 03 pipeline, '
        'or set `USE_EXISTING_EXPERIMENT8_ARTIFACTS = True` only when you intentionally want old sandbox artifacts.'
    ))
